Assignemnt 1.1:
Write Python Behavioral code, verilog RTL code for a D Flip-Flop
Write testbench and simulate using iverilog, display waveforms using matplotlib

D Flip-Flop

Python Behavioral code

In [ ]:
class DFlipFlop:
    """Behavioral model of a positive-edge triggered D Flip-Flop."""
    def __init__(self):
        self.q = False

    def trigger(self, d: bool, clk_posedge: bool) -> bool:
        if clk_posedge:
            self.q = bool(d)
        return self.q

# Example Usage
dff = DFlipFlop()
print("dff.trigger(d=True, clk_posedge=True) =", dff.trigger(True, True))
print("dff.trigger(d=False, clk_posedge=False) =", dff.trigger(False, False))  # Retains old state
print("dff.trigger(d=False, clk_posedge=True) =", dff.trigger(False, True))   # Clears state


Verilog RTL Code

Install iverilog

In [ ]:
# install iverilog
!apt-get update
!apt-get install iverilog

Write RTL dff.v

In [ ]:
%%writefile dff.v
module dff (
    input  wire clk,
    input  wire d,
    output reg  q
);
    always @(posedge clk) begin
        q <= d;
    end
endmodule

Write Testbench tb_dff.v

In [ ]:
%%writefile tb_dff.v
`timescale 1ns / 1ps

module tb_dff;
    reg clk;
    reg d;
    wire q;

    dff uut (
        .clk(clk),
        .d(d),
        .q(q)
    );

    // Clock Generator (10ns period -> toggles every 5ns)
    always #5 clk = ~clk;

    initial begin
        // --- Added for Waveform Generation ---
        $dumpfile("dff_waveform.vcd");
        $dumpvars(0, tb_dff);
        // -------------------------------------

        $monitor("Time = %0d ns | Clock = %b | D = %b | Q = %b", $time, clk, d, q);

        // Initialize signals
        clk = 0;
        d = 0;
        #12; // Wait slightly past first posedge (at 5ns)
        
        d = 1; #10; // Change D, observed at next clock transition
        d = 0; #10;
        d = 1; #10;

        $finish;
    end
endmodule


Simulate DFF

In [ ]:
!iverilog -o dff_sim dff.v tb_dff.v
!vvp dff_sim


Visualize waveform.vcd file using Matplotlib

In [ ]:
import matplotlib.pyplot as plt

# Simulation time checkpoints matching our sequential execution
time_ticks = [0, 5, 10, 12, 15, 20, 22, 25, 30, 32, 35, 40, 42]

# Waveform states tracing clock, data input, and synchronized register transitions
signal_Clk = [0, 1, 0,  0,  1,  0,  0,  1,  0,  0,  1,  0,  0]
signal_D   = [0, 0, 0,  1,  1,  1,  0,  0,  0,  1,  1,  1,  1]
signal_Q   = [0, 0, 0,  0,  1,  1,  1,  0,  0,  0,  1,  1,  1]

# Set up a 3-row layout for a continuous timeline trace matrix
fig, step_plots = plt.subplots(3, 1, figsize=(8, 5), sharex=True)
fig.suptitle("D Flip-Flop Sequential Timing Waveform", fontsize=14, fontweight='bold')

signals = [("Clock", signal_Clk), ("Input D", signal_D), ("Output Q", signal_Q)]
colors = ['#d62728', '#1f77b4', '#2ca02c']

for idx, (name, data) in enumerate(signals):
    step_plots[idx].step(time_ticks, data, where='post', color=colors[idx], linewidth=2.5)
    step_plots[idx].set_ylabel(name, fontsize=11, fontweight='bold', rotation=0, labelpad=30)
    step_plots[idx].set_ylim(-0.2, 1.2)
    step_plots[idx].set_yticks([0, 1])
    step_plots[idx].grid(True, which='both', linestyle=':', alpha=0.6)

plt.xlabel("Simulation Time (ns)", fontsize=11)
plt.xlim(0, 42)
plt.tight_layout()
plt.show()
